In [4]:
import os
import pandas as pd
import numpy as np
import re # Import regular expressions for extracting numbers

path_to_scenarios = '/home/abhis/india_power/scenarios'
# Get all directory names within path_to_scenarios
# all_entries = os.listdir(path_to_scenarios)
# Filter for directories only
# scenario_names = [entry for entry in all_entries if os.path.isdir(os.path.join(path_to_scenarios, entry))]
scenario_names = ['ra_2050_1', 'ra_2050_2'] # Reverted to explicit list
print(f"Processing scenarios: {scenario_names}")
# path_to_images    = '/home/abhis/india_power/images'
# path_to_csvs      = '/home/abhis/india_power/gridpath_india_viz/csvs'

all_load_data = []

for scenario_name in scenario_names:
    scenario_path = os.path.join(path_to_scenarios, scenario_name)
    # No need to check isdir again, already filtered
    # if not os.path.isdir(scenario_path):
    #     print(f"Warning: Scenario directory not found: {scenario_path}")
    #     continue

    # Iterate through weather iterations
    for weather_folder in os.listdir(scenario_path):
        if weather_folder.startswith('weather_iteration_'):
            weather_match = re.search(r'\d+', weather_folder)
            if not weather_match:
                continue
            weather_iteration = int(weather_match.group())
            weather_path = os.path.join(scenario_path, weather_folder)
            if not os.path.isdir(weather_path):
                continue

            # Iterate through hydro iterations
            for hydro_folder in os.listdir(weather_path):
                if hydro_folder.startswith('hydro_iteration_'):
                    hydro_match = re.search(r'\d+', hydro_folder)
                    if not hydro_match:
                        continue
                    hydro_iteration = int(hydro_match.group())
                    hydro_path = os.path.join(weather_path, hydro_folder)
                    if not os.path.isdir(hydro_path):
                        continue

                    # Iterate through availability iterations
                    for avail_folder in os.listdir(hydro_path):
                        if avail_folder.startswith('availability_iteration_'):
                            avail_match = re.search(r'\d+', avail_folder)
                            if not avail_match:
                                continue
                            availability_iteration = int(avail_match.group())
                            avail_path = os.path.join(hydro_path, avail_folder)
                            if not os.path.isdir(avail_path):
                                continue

                            # Construct path to load_mw.tab
                            load_tab_path = os.path.join(avail_path, 'inputs', 'load_mw.tab') # Corrected filename

                            if os.path.exists(load_tab_path):
                                try:
                                    # Read the load_mw.tab file
                                    df_load = pd.read_csv(load_tab_path, sep='\t')

                                    # Add iteration columns
                                    df_load['scenario_name'] = scenario_name
                                    df_load['weather_iteration'] = weather_iteration
                                    df_load['hydro_iteration'] = hydro_iteration
                                    df_load['availability_iteration'] = availability_iteration

                                    # Reorder columns to put iteration columns first
                                    iteration_cols = ['scenario_name', 'weather_iteration', 'hydro_iteration', 'availability_iteration']
                                    original_cols = [col for col in df_load.columns if col not in iteration_cols]
                                    df_load = df_load[iteration_cols + original_cols]

                                    all_load_data.append(df_load)
                                except Exception as e:
                                    print(f"Error reading {load_tab_path}: {e}")
                            else:
                                print(f"Warning: load_mw.tab not found in {os.path.join(avail_path, 'inputs')}") # Corrected filename in warning

# Concatenate all dataframes
if all_load_data:
    final_load_df = pd.concat(all_load_data, ignore_index=True)
    print("Combined DataFrame created successfully.")
    # Display the first few rows (optional)
    # print(final_load_df.head())
else:
    print("No load_mw.tab files were found or processed.") # Corrected filename in message

Processing scenarios: ['ra_2050_1', 'ra_2050_2']
Combined DataFrame created successfully.
Combined DataFrame created successfully.
